# E3 — Contact onset over approach velocity

`e3_approach.py`. Contact metrics per run, averaged over its approaches: force-onset latency after the
leader crosses the table, leader penetration at onset, felt overshoot, robot peak table force, e_F during
the hold. Expected: all methods converge at low speed; M1's onset kick grows with v·T; the model-based
transients scale with momentum (Λv for M5, mv for M4, none for M3).

In [ ]:
# Runs to analyse: run directories, session files (exp/<exp>/sessions/*.txt, written by the script) or a
# folder of runs (a folder also holds retried and superseded runs). Default: the latest session file,
# which lists every condition once.
from common import latest_session

RUNS = [latest_session("e3")]
RECOMPUTE = False  # True: recompute the metrics from the data instead of reading analysis/metrics.json
SWEEP_RTT = "850"      # the delay of the speed sweep
SWEEP_SPEED = 0.05     # the speed of the delay sweep [m/s]

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()  # experiments/rim-delays/exp
sys.path[:0] = [str(HERE), str(HERE.parent)]
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import exp_metrics
import plots as P
from common import collect, load_table

%matplotlib inline
pd.set_option("display.precision", 3)

runs = collect(RUNS)
df = load_table(runs, exp_metrics.e3 if RECOMPUTE else None)
print(f"{len(df)} runs")

In [ ]:
df["T"] = df.rtt.map(P.delay_key)
cols = ["run", "method", "T", "approach_speed", "contacts", "onset_latency_ms_mean", "penetration_at_onset_mm_mean",
        "overshoot_mean", "robot_peak_table_N_mean", "eF_z_mean_mean", "hf_rms_N_mean", "E_min_J"]
df[cols].sort_values(["T", "approach_speed", "method"])

## Against the approach speed (at `SWEEP_RTT`)

In [ ]:
metrics = [("onset_latency_ms_mean", "force-onset latency [ms]"), ("penetration_at_onset_mm_mean", "penetration at onset [mm]"),
           ("overshoot_mean", "felt overshoot (peak / settled)"), ("robot_peak_table_N_mean", "robot peak table force [N]")]
sub = df[df["T"] == SWEEP_RTT]
fig, axes = plt.subplots(1, len(metrics), figsize=(4.2 * len(metrics), 3.3))
for ax, (y, title) in zip(axes, metrics):
    for m in P.methods_in(sub):
        s = sub[sub.method == m].groupby("approach_speed")[y].mean()
        ax.plot(100 * s.index, s.values, "o-", color=P.color(m), label=P.LABELS.get(m, m) if ax is axes[0] else None)
    ax.set_title(title, loc="left", fontsize=10)
    ax.set_xlabel("approach speed [cm/s]")
fig.suptitle(f"T = {SWEEP_RTT} ms", x=0.01, ha="left")
fig.legend(loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.1))
fig.tight_layout()

## Against the delay (at `SWEEP_SPEED`)

In [ ]:
sub = df[np.isclose(df.approach_speed.astype(float), SWEEP_SPEED)]
fig, axes = plt.subplots(1, len(metrics), figsize=(4.2 * len(metrics), 3.3))
for ax, (y, title) in zip(axes, metrics):
    P.by_delay(ax, sub, y, label_methods=(ax is axes[0]))
    ax.set_title(title, loc="left", fontsize=10)
fig.suptitle(f"approach at {100 * SWEEP_SPEED:g} cm/s", x=0.01, ha="left")
fig.legend(loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.1))
fig.tight_layout()

## One contact, M1 vs M5 (at `SWEEP_RTT`, `SWEEP_SPEED`)
Leader height, felt force, and the robot's table force when the operator's action reached it.

In [ ]:
import metrics as M
from runs import load_run, surface_of

pick = df[(df["T"] == SWEEP_RTT) & np.isclose(df.approach_speed.astype(float), SWEEP_SPEED)]
fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
for method in ("zoh", "proxy-rim"):
    rows = pick[pick.method == method]
    if rows.empty:
        continue
    run = load_run(rows.iloc[0].dir)
    first = P.metrics_of(rows.iloc[0].dir)["per_contact"][0]["t_cross"]
    e = M.unfelt_force(run)
    hs = M.haptic_side(run)
    w = (e["t"] > first - 0.5) & (e["t"] < first + 2.5)
    t = e["t"][w] - first
    axes[0].plot(t, 1e3 * (hs["x_l"][w, 2] - surface_of(run["meta"])), color=P.color(method), label=P.LABELS[method])
    axes[1].plot(t, e["f"][w, 2], color=P.color(method), label=f"{P.LABELS[method]}: felt")
    axes[1].plot(t, -e["F_osc"][w, 2], color=P.color(method), ls="--", label=f"{P.LABELS[method]}: robot table force")
axes[0].axhline(0, color=P.INK_2, lw=1)
axes[0].set_ylabel("leader height above the table [mm]")
axes[1].set_ylabel("force, z [N]")
axes[1].set_xlabel("time from the leader's crossing [s]")
axes[0].legend()
axes[1].legend(fontsize=8)
fig.tight_layout()